# Image Captioning with ViT and Llama (LoRA)

This notebook implements an image captioning system using:
- Vision Transformer (ViT) for image features
- Llama-3.2-1B with LoRA for text generation
- Projection layer to connect visual and text features

In [15]:
# Standard library imports
import os
import sys
from datetime import datetime

# Third-party imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    ViTImageProcessor, 
    ViTModel,
    AutoTokenizer, 
    AutoModelForCausalLM
)
from peft import LoraConfig, get_peft_model
from dotenv import load_dotenv
import wandb
from tqdm.notebook import tqdm

# Local imports
sys.path.insert(0, "/ghome/c5mcv07/C5_G7_MCV")
from Image_Captioning_Utils.metrics import calculate_metrics
from Image_Captioning_Utils.dataset import FoodDatasetWord
from Image_Captioning_Utils.utils import get_train_val_test_annotations_split
from Image_Captioning_Utils.constants import TEXT_MAX_LEN

In [16]:
# Configuration and Setup
load_dotenv()
device = "cuda" if torch.cuda.is_available() else "cpu"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
experiment_name = f"{timestamp}_MODEL_TRAINING"

In [17]:
class Projection(nn.Module):
    def __init__(self, vit_hidden_size=768, llama_hidden_size=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(vit_hidden_size, llama_hidden_size),
            nn.GELU(),
            nn.LayerNorm(llama_hidden_size)
        )

    def forward(self, x):
        return self.proj(x)

In [52]:
def initialize_models():
    # ViT Model
    print("Initializing ViT model...")
    vit_model_name = "google/vit-base-patch16-224-in21k"
    processor = ViTImageProcessor.from_pretrained(vit_model_name)
    vit_model = ViTModel.from_pretrained(vit_model_name).to(device)
    
    # Freeze ViT parameters
    for param in vit_model.parameters():
        param.requires_grad = False
        
    # Llama Model with LoRA
    print("Initializing Llama model with LoRA...")
    llama_model_name = "meta-llama/Llama-3.2-1B"
    tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        llama_model_name,
        device_map=device,
        torch_dtype=torch.float32
    )
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    
    # Projection layer
    print("Initializing projection layer...")
    projection_layer = Projection(
        vit_hidden_size=768,
        llama_hidden_size=model.config.hidden_size
    ).to(device)
    
    return processor, vit_model, tokenizer, peft_model, projection_layer

In [53]:
# Test model initialization
processor, vit_model, tokenizer, peft_model, projection_layer = initialize_models()
print("Model initialization successful!")

Initializing ViT model...
Initializing Llama model with LoRA...
trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689
Initializing projection layer...
Model initialization successful!


In [64]:
def create_data_loaders(processor, tokenizer):
    def custom_collate_fn(batch):
        images, captions = zip(*batch)
        
        pixel_values = processor(images=list(images), return_tensors="pt").pixel_values.squeeze()

        tokenized = tokenizer(
            list(captions),
            padding="max_length",
            truncation=True,
            max_length=200,
            return_tensors="pt"
        )

        cap_idxs = tokenized["input_ids"]

        return {
            "pixel_values": pixel_values,
            "captions": captions,
            "cap_idxs": cap_idxs,
        }

    splits = get_train_val_test_annotations_split()
    train_dataset = FoodDatasetWord(splits["train"])
    val_dataset = FoodDatasetWord(splits["val"])

    training_config = {
        "lr": 1e-4,
        "batch_size": 40,
        "epochs": 10,
        "weight_decay": 1e-3,
        "num_workers": 1
    }

    train_loader = DataLoader(
        train_dataset, 
        batch_size=training_config['batch_size'], 
        shuffle=True, 
        collate_fn=custom_collate_fn, 
        num_workers=training_config["num_workers"], 
        pin_memory=True, 
        persistent_workers=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=training_config['batch_size'], 
        shuffle=False, 
        collate_fn=custom_collate_fn, 
        num_workers=training_config["num_workers"], 
        pin_memory=True, 
        persistent_workers=True
    )
    
    return train_loader, val_loader, training_config

In [65]:
# Test data loading
train_loader, val_loader, training_config = create_data_loaders(processor, tokenizer)
sample_batch = next(iter(train_loader))
print("Data loading successful!")
print(f"Batch keys: {sample_batch.keys()}")
print(f"Pixel values shape: {sample_batch['pixel_values'].shape}")
print(f"Captions length: {len(sample_batch['captions'])}")
print(f"Caption indices shape: {sample_batch['cap_idxs'].shape}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Data loading successful!
Batch keys: dict_keys(['pixel_values', 'captions', 'cap_idxs'])
Pixel values shape: torch.Size([40, 3, 224, 224])
Captions length: 40
Caption indices shape: torch.Size([40, 200])


In [66]:
def setup_training(model, training_config):
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=training_config['lr'],  
        weight_decay=training_config['weight_decay']
    )
    num_epochs = training_config['epochs']
    return optimizer, num_epochs

In [67]:
# Test training setup
optimizer, num_epochs = setup_training(peft_model, training_config)
print("Training setup successful!")
print(f"Optimizer: {optimizer}")
print(f"Number of epochs: {num_epochs}")

Training setup successful!
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.001
)
Number of epochs: 10


In [75]:
# Test with a single batch before full training
print("Testing with a single batch...")
test_batch = next(iter(train_loader))
pixel_values = test_batch["pixel_values"].to(device)
labels = test_batch["cap_idxs"].to(device)

# Process through ViT
with torch.no_grad():
    vit_outputs = vit_model(pixel_values=pixel_values)
    image_features = vit_outputs.last_hidden_state[:, 0, :]

# Project features
visual_embeds = projection_layer(image_features)

# Get text embeddings
text_embeds = peft_model.get_input_embeddings()(labels)

pixel_values.shape, labels.shape, image_features.shape, visual_embeds.shape, text_embeds.shape

Testing with a single batch...


(torch.Size([40, 3, 224, 224]),
 torch.Size([40, 200]),
 torch.Size([40, 768]),
 torch.Size([40, 2048]),
 torch.Size([40, 200, 2048]))

In [76]:
# Combine embeddings
inputs_embeds = torch.cat([
    visual_embeds.unsqueeze(1),
    text_embeds[:, :-1, :]
], dim=1)

attention_mask = (labels != tokenizer.pad_token_id).float()

inputs_embeds.shape, attention_mask.shape

(torch.Size([40, 200, 2048]), torch.Size([40, 200]))

In [ ]:
# Test forward pass
outputs = peft_model(
    inputs_embeds=inputs_embeds,
    labels=labels,
    attention_mask=attention_mask
)

print("Single batch test successful!")
print(f"Output loss: {outputs.loss.item()}")

In [ ]:
def train_model(model, tokenizer, train_loader, val_loader, optimizer, num_epochs, vit_model, projection_layer):
    # Initialize wandb
    wandb.login(key=os.getenv('WANDB_GERARD'))
    wandb.init(name=experiment_name, project="C5-G7-LLMs", config=training_config)
    wandb.watch(model, log="all", log_freq=100)

    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            optimizer.zero_grad()
            
            # Process images
            pixel_values = batch["pixel_values"].to(device)
            with torch.no_grad():
                image_features = vit_model(pixel_values=pixel_values).last_hidden_state[:, 0, :]
            
            # Prepare inputs
            visual_embeds = projection_layer(image_features)
            labels = batch["cap_idxs"].to(device)
            text_embeds = model.get_input_embeddings()(labels).to(device)
            
            # Combine embeddings
            inputs_embeds = torch.cat([
                visual_embeds.unsqueeze(1),
                text_embeds[:, :-1, :]
            ], dim=1)
            
            # Create attention mask
            attention_mask = (labels != tokenizer.pad_token_id).float().to(device)
            
            # Forward pass
            outputs = model(
                inputs_embeds=inputs_embeds,
                labels=labels,
                attention_mask=attention_mask
            )
            
            # Backward pass
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            # Log training progress
            wandb.log({"train_loss": loss.item()})

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}"):
                # Process images
                pixel_values = batch["pixel_values"].to(device)
                image_features = vit_model(pixel_values=pixel_values).last_hidden_state[:, 0, :]
                
                # Prepare inputs
                visual_embeds = projection_layer(image_features)
                labels = batch["cap_idxs"].to(device)
                text_embeds = model.get_input_embeddings()(labels).to(device)
                
                # Combine embeddings
                inputs_embeds = torch.cat([
                    visual_embeds.unsqueeze(1),
                    text_embeds[:, :-1, :]
                ], dim=1)
                
                # Create attention mask
                attention_mask = (labels != tokenizer.pad_token_id).float().to(device)
                
                # Forward pass
                outputs = model(
                    inputs_embeds=inputs_embeds,
                    labels=labels,
                    attention_mask=attention_mask
                )
                val_loss += outputs.loss.item()

        # Calculate epoch metrics
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        # Log metrics
        wandb.log({
            "epoch": epoch+1,
            "avg_train_loss": avg_train_loss,
            "val_loss": avg_val_loss
        })
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'projection_state_dict': projection_layer.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'loss': best_val_loss,
            }, f'checkpoints/gpt2/{experiment_name}_best_model.pth')